# The Anatomy of AI · Lecture 2 · Learning
### The notebook

In Lecture 1 we set the walk neuron's three numbers by hand: weights 2.0 and −3.0, bias −0.5. Real networks have billions
of numbers and nobody sets them. They are **learned** from examples. This notebook makes one neuron learn its own numbers
from Lecture 1's diary, with nothing but NumPy, and checks every result against the lecture.

Each section names the lecture plate it rebuilds. A `check(...)` line prints ✓ when your number matches the lecture's, and
stops the notebook if it does not. Five **Your turn** exercises sit along the way; solutions are at the end.

It runs on free Colab in about a minute, on the CPU. No GPU, no downloads, no keys.

In [ ]:
# Pinned to the versions free Colab ships (Python 3.12); nothing else is needed, and no keys.
%pip install -q numpy==2.0.2 matplotlib==3.10.0

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# The course's look: black ink on paper, no colour.
PAPER, INK, INK2 = "#F1EBDD", "#16130F", "#7A7062"
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "font.family": "serif", "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "hatch.color": INK, "hatch.linewidth": 0.6, "figure.dpi": 110, "legend.frameon": False,
})
INKMAP = LinearSegmentedColormap.from_list("ink", [PAPER, INK])   # a value as ink density, as on the plates


def minus(text):
    """Print negative numbers with a real minus sign, as the lecture does."""
    return str(text).replace("-", "−")


def check(name, value, expected, tol=1e-9):
    """Compare a result with the lecture's own number. Prints ✓, or stops the notebook with ✗."""
    if isinstance(expected, (int, float)) and not isinstance(expected, bool):
        ok = abs(float(value) - expected) <= tol
        shown = f"{float(value):.6g}"
    else:
        ok = value == expected
        shown = repr(value)
    want = f"{expected:.6g}" if isinstance(expected, float) else expected
    print(f"{'✓' if ok else '✗'} {name}: {shown}   (lecture: {want})")
    if not ok:
        raise AssertionError(f"{name}: got {shown}, the lecture says {expected}")

## The diary and the neuron, from Lecture 1

The diary is Lecture 1's, exactly: 42 days drawn with Python's random number generator (seed 11), 18 enjoyed walks and 24
not. The neuron is Lecture 1's too: multiply and add, then the gate.

We keep all 42 days in two arrays: `X` holds (warmth, rain) for each day, `Y` holds 1 for an enjoyed walk and 0 for one you
did not.

In [ ]:
import random

def make_diary(rule, seed, good, bad):
    rng = random.Random(seed)
    days, need = [], {True: good, False: bad}
    while need[True] or need[False]:
        w, r = rng.uniform(0.03, 0.97), rng.uniform(0.03, 0.97)
        label = rule(w, r)
        if label is None or not need[label]:
            continue
        need[label] -= 1
        days.append((w, r, label))
    return days

def lecture1_rule(w, r):
    t = 2.0 * w - 3.0 * r - 0.5
    return None if abs(t) < 0.18 else t > 0

diary = make_diary(lecture1_rule, seed=11, good=18, bad=24)
X = np.array([(w, r) for w, r, _ in diary])          # 42 × 2: warmth, rain
Y = np.array([1.0 if e else 0.0 for *_, e in diary])  # 1 = enjoyed
check("days", len(X), 42)
check("enjoyed", int(Y.sum()), 18)

## Plate I · Where we left off

Who chose Lecture 1's numbers? We did, by hand. This lecture shows how a neuron finds them itself.

## Plate II · The recipe

Learning is a loop:

1. **Guess** the numbers.
2. **Try** them on a day.
3. **Measure** how wrong the answer is.
4. **Find** which way to turn each number.
5. **Turn** each one a little. Then go round again.

## Plate III · A starting guess

Real networks start from small random numbers. We start from round ones, to keep the arithmetic easy: both weights 1.0,
bias −0.5. To say how good a guess is we need two more ideas.

## Plate IV · What we want

For each day we write down the answer we want, its **target**: 1 for a walk we enjoyed, 0 for one we did not.

- The **error** on one day is (output − target)²: zero when the answer is exactly right, bigger the further off it is.
  (Practitioners call this the *loss*.)
- The **average error** over the diary is the one number learning will push down.
- The neuron's **verdict** is *go* when its output is nearer 1 than 0, that is, above ½. A day is **right** when the verdict
  matches what happened.

In [ ]:
def forward(w, b, X=X):
    """Lecture 1's neuron on every day at once. w = (weight on warmth, weight on rain)."""
    total = X @ w + b
    return total, np.maximum(0.0, total)

def average_error(w, b, X=X, Y=Y):
    _, out = forward(w, b, X)
    return float(np.mean((out - Y) ** 2))

def days_right(w, b, X=X, Y=Y):
    _, out = forward(w, b, X)
    return int(np.sum((out > 0.5) == (Y > 0)))

guess_w, guess_b = np.array([1.0, 1.0]), -0.5
check("the guess: days right", days_right(guess_w, guess_b), 17)
check("the guess: average error", average_error(guess_w, guess_b), 0.4357, tol=5e-5)

In [ ]:
def diary_plot(ax, w, b, X=X, Y=Y, title=None):
    """The diary, hatched where the neuron says go (output above ½), its wrong days ringed."""
    g = np.linspace(0, 1, 201)
    Wg, Rg = np.meshgrid(g, g)
    T = w[0] * Wg + w[1] * Rg + b
    if (T > 0.5).any():
        ax.contourf(Wg, Rg, T, levels=[0.5, max(T.max(), 0.51)], colors="none", hatches=["////"])
    ax.contour(Wg, Rg, T, levels=[0.5], colors=INK, linewidths=2)
    _, out = forward(w, b, X)
    for (x, r), y, o in zip(X, Y, out):
        ax.plot(x, r, "o", ms=6, mfc=INK if y else PAPER, mec=INK, mew=1.1)
        if (o > 0.5) != (y > 0):
            ax.plot(x, r, "o", ms=14, mfc="none", mec=INK, mew=0.9)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_aspect("equal")
    ax.set_xlabel("warmth"); ax.set_ylabel("rain")
    if title: ax.set_title(title, fontsize=10)

fig, ax = plt.subplots(figsize=(4.2, 4.2))
diary_plot(ax, guess_w, guess_b, title="the guess: hatched where it says go; wrong days ringed")
plt.show()

## Plate V · One day

Try the guess on the rainy day from Lecture 1: warmth 0.8, rain 0.6, a walk we did not enjoy (target 0).

In [ ]:
rainy, target = np.array([0.8, 0.6]), 0.0
total = rainy @ guess_w + guess_b
output = max(0.0, total)
err = (output - target) ** 2
print(f"total {total:.2f}, output {output:.2f}, wanted {target:.0f}, error {err:.2f}")
check("rainy day: output", output, 0.9)
check("rainy day: error", err, 0.81)

## Plate VI · The nudge test

Which way should the weight on rain move? Nudge it a little and watch the error. If the error rises, the other way is
downhill. **Rise ÷ nudge** is the **slope**: how fast the error grows as the weight grows.

In [ ]:
def rainy_error(w_rain, w_warmth=1.0, b=-0.5):
    return max(0.0, 0.8 * w_warmth + 0.6 * w_rain + b) ** 2

rise = rainy_error(1.01) - rainy_error(1.0)
print(f"nudge the rain weight to 1.01: the total becomes {0.8 + 0.606 - 0.5:.3f}, the error {rainy_error(1.01):.4f}")
print(f"rise {rise:.4f}; rise ÷ nudge = {rise / 0.01:.4f}")
check("the error rises by", rise, 0.0108, tol=1e-4)
check("rise ÷ nudge", rise / 0.01, 1.08, tol=5e-3)
check("with a smaller nudge (0.001)", (rainy_error(1.001) - rainy_error(1.0)) / 0.001, 1.0804, tol=5e-5)

In [ ]:
ws = np.linspace(-1, 1.6, 300)
fig, ax = plt.subplots(figsize=(5.6, 3.4))
ax.plot(ws, [rainy_error(v) for v in ws], color=INK, lw=2)
ax.plot([1.0], [rainy_error(1.0)], "o", color=INK)
ax.plot(ws, rainy_error(1.0) + 1.08 * (ws - 1.0), color=INK, lw=0.9, ls="--")
ax.annotate("the guess: slope 1.08", (1.0, 0.81), (0.0, 1.2), arrowprops=dict(arrowstyle="-", color=INK2))
ax.set_ylim(-0.05, 1.6); ax.set_xlabel("weight on rain"); ax.set_ylabel("error on the rainy day")
plt.show()

## Plate VII · The slope in three factors

We do not have to nudge. The slope is a product of three small questions:

1. How fast does the error grow with the answer? **Twice the miss**: 2 × (output − target).
2. How fast does the answer grow with the total? **The gate**: 1 if the neuron fires, 0 if it is silent.
3. How fast does the total grow with this weight? **The input** on that wire (for the bias, 1).

Multiply them: slope = 2 × miss × gate × input.

In [ ]:
def slopes_one_day(day, target, w, b):
    """The three-factor chain for one day: slopes of its error for (warmth weight, rain weight, bias)."""
    total = day @ w + b
    out = max(0.0, total)
    gate = 1.0 if total > 0 else 0.0
    twice_miss = 2 * (out - target)
    return twice_miss * gate * day[0], twice_miss * gate * day[1], twice_miss * gate * 1.0

s_warm, s_rain, s_bias = slopes_one_day(rainy, 0.0, guess_w, guess_b)
print(f"rain:   2 × 0.9 × 1 × 0.6 = {s_rain:.2f}")
print(f"warmth: 2 × 0.9 × 1 × 0.8 = {s_warm:.2f}")
check("slope for the rain weight", s_rain, 1.08)
check("slope for the warmth weight", s_warm, 1.44)
check("slope for the bias", s_bias, 1.8)

## Plate VIII · A shut gate learns nothing

A cool, dry day we enjoyed: warmth 0.3, rain 0.1. The guess gives a total of −0.1, so the gate is shut and the answer is 0.
The error is big (we wanted 1), yet every slope is 0: a small nudge leaves the gate shut and the error unchanged.

In [ ]:
cool = np.array([0.3, 0.1])
total_cool = cool @ guess_w + guess_b
print(f"total {total_cool:+.2f}; error {(max(0, total_cool) - 1) ** 2:.1f}; slopes {slopes_one_day(cool, 1.0, guess_w, guess_b)}")
check("cool dry day: total", total_cool, -0.1)
check("cool dry day: every slope", max(abs(v) for v in slopes_one_day(cool, 1.0, guess_w, guess_b)), 0.0)

## Plate IX · One step downhill

Turn each weight a little *against* its slope: **new = old − step size × slope**. The **step size** says how big "a
little" is; here 0.1. (We hold the bias at −0.5 for now; it gets its turn on Plate XVII.)

In [ ]:
lr = 0.1
new_w = guess_w - lr * np.array([s_warm, s_rain])
new_total = rainy @ new_w + guess_b
print(f"rain:   1.0 − 0.1 × 1.08 = {new_w[1]:.3f}")
print(f"warmth: 1.0 − 0.1 × 1.44 = {new_w[0]:.3f}")
print(f"the rainy day again: total {new_total:.2f}, error {new_total ** 2:.4f} (was 0.81)")
check("new rain weight", new_w[1], 0.892)
check("new warmth weight", new_w[0], 0.856)
check("rainy day after one step: total", new_total, 0.72)
check("rainy day after one step: error", new_total ** 2, 0.5184)

(Plate X is the narrated film of these steps.)

### Your turn 1 · The sunny day by hand

Lecture 1's sunny day: warmth 0.8, rain 0.1, a walk we enjoyed (target 1). What does the guess answer, and what is the
error?

*Hint:* total = 0.8 × 1.0 + 0.1 × 1.0 − 0.5.

In [ ]:
sunny_output, sunny_error = None, None   # TODO

### Your turn 2 · Its slope

What is the sunny day's slope for the warmth weight, and which way does one step move that weight?

*Hint:* use `slopes_one_day(np.array([0.8, 0.1]), 1.0, guess_w, guess_b)`; a negative slope means the step turns the weight up.

In [ ]:
sunny_slope_warmth = None   # TODO

## Plate XI · Days disagree

The rainy day pushes both weights down; the sunny day pulls both up. One day is a narrow view.

## Plate XII · All 42 days at once

So we average the slopes over every day and take one step on the average. On the first step, a surprise: the warmth
weight goes *down*, a little. The false alarms (warm rainy days we did not enjoy, which the guess calls *go*) push it down
harder than the enjoyed days pull it up.

In [ ]:
def average_slopes(w, b, X=X, Y=Y):
    """The three-factor chain on every day, averaged: slopes for (warmth weight, rain weight) and for the bias."""
    total, out = forward(w, b, X)
    g = 2 * (out - Y) * (total > 0)
    return (g[:, None] * X).mean(axis=0), float(g.mean())

gw, _ = average_slopes(guess_w, guess_b)
total, out = forward(guess_w, guess_b)
per_day = 2 * (out - Y) * (total > 0) * X[:, 0] / len(X)          # each day's share of the warmth slope
push_down, pull_up = per_day[Y == 0].sum(), -per_day[Y == 1].sum()
print(f"average slopes: warmth {gw[0]:+.4f}, rain {gw[1]:+.4f}")
print(f"warmth: the days we did not enjoy push down {push_down:.4f}; the enjoyed days pull up {pull_up:.4f}")
check("average slope, warmth", gw[0], 0.0186, tol=5e-5)
check("average slope, rain", gw[1], 0.3469, tol=5e-5)
check("false alarms push warmth down by", push_down, 0.3828, tol=5e-5)
check("enjoyed days pull warmth up by", pull_up, 0.3642, tol=5e-5)

after = guess_w - 1.0 * gw                      # one step, step size 1
print(f"after one step (size 1): weights {after.round(4)}, error {average_error(after, guess_b):.4f}, right {days_right(after, guess_b)}")
check("after one step: warmth weight", after[0], 0.9814, tol=5e-5)
check("after one step: rain weight", after[1], 0.6531, tol=5e-5)
check("after one step: average error", average_error(after, guess_b), 0.3406, tol=5e-5)
check("after one step: days right", days_right(after, guess_b), 19)

In [ ]:
fig, ax = plt.subplots(figsize=(4.6, 4.6))
steps = -(2 * (out - Y) * (total > 0))[:, None] * X          # each day's own step, at step size 1
for (dx, dy), y in zip(steps * 0.25, Y):
    ax.annotate("", xy=(1 + dx, 1 + dy), xytext=(1, 1), arrowprops=dict(arrowstyle="-|>", color=INK if y else INK2,
                lw=1.0, ls="-" if y else "--"))
ax.annotate("", xy=tuple(after), xytext=(1, 1), arrowprops=dict(arrowstyle="-|>", color=INK, lw=3))
ax.plot([1], [1], "o", color=INK)
ends = np.vstack([1 + steps * 0.25, after, [[1, 1]]])            # annotations don't widen the axes, so set them
pad = 0.08
ax.set_xlim(ends[:, 0].min() - pad, ends[:, 0].max() + pad); ax.set_ylim(ends[:, 1].min() - pad, ends[:, 1].max() + pad)
ax.set_xlabel("weight on warmth"); ax.set_ylabel("weight on rain"); ax.set_aspect("equal")
ax.set_title("each day's step (a quarter size; dashed: not enjoyed)\nand their average, full size", fontsize=10)
plt.show()

## Plate XIII · The valley

Do this for every pair of weights and you get a landscape: the average error as height, over the two weights (bias held
at −0.5). We call it the **valley**. Two landmarks:

- **the floor**, the lowest point;
- **a flat plain** where the warmth weight is small: there the neuron is silent on every day, says 0 to every enjoyed day,
  and the error is exactly 18 ÷ 42.

In [ ]:
wa, ra = np.linspace(-2, 6, 161), np.linspace(-8, 2, 201)
E = np.array([[average_error(np.array([a, c]), -0.5) for a in wa] for c in ra])

def descend(w, b, lr, steps, learn_bias=False):
    """Step downhill on the average error, `steps` times. Returns the final numbers and the path of weights."""
    w, b, path = np.array(w, float), float(b), [np.array(w, float)]
    for _ in range(steps):
        gw, gb = average_slopes(w, b)
        w = w - lr * gw
        if learn_bias:
            b = b - lr * gb
        path.append(w.copy())
    return w, b, np.array(path)

floor_w, _, _ = descend(guess_w, -0.5, 1.0, 20000)
floor_err = round(average_error(floor_w, -0.5), 5)
print(f"the floor: weights {floor_w.round(4)}, error {floor_err}, right {days_right(floor_w, -0.5)} of 42")
check("the plain's error", average_error(np.array([0.0, 0.0]), -0.5), 18 / 42)
check("the plain, as the lecture prints it", round(18 / 42, 4), 0.4286)
check("floor: warmth weight", floor_w[0], 2.3165, tol=5e-5)
check("floor: rain weight", floor_w[1], -2.0032, tol=5e-5)
check("floor: average error", floor_err, 0.04141)
check("floor: days right", days_right(floor_w, -0.5), 41)

In [ ]:
def valley(ax, path=None):
    ax.contour(wa, ra, np.minimum(E, 1.0), levels=[0.05, 0.08, 0.12, 0.2, 0.3, 0.42, 0.6, 0.8], colors=INK, linewidths=0.7)
    flat = np.isclose(E, 18 / 42)
    ax.contourf(wa, ra, flat.astype(float), levels=[0.5, 1.5], colors="none", hatches=["...."])
    ax.plot(*floor_w, "x", color=INK, ms=9, mew=2)
    if path is not None:
        ax.plot(path[:, 0], path[:, 1], "-o", color=INK, lw=1, ms=2.5)
    ax.set_xlabel("weight on warmth"); ax.set_ylabel("weight on rain")

fig, ax = plt.subplots(figsize=(5.2, 5.2))
valley(ax)
ax.text(-1.6, -7.2, "the flat plain (dotted):\nsilent on every day", fontsize=9)
ax.set_title("the valley from above; × the floor", fontsize=10)
plt.show()

## Plate XIV · Rolling down

Start at the guess and step downhill again and again, step size 1, 120 steps. The ball rolls down the valley, and on the
diary its line swings round.

In [ ]:
run_w, _, run_path = descend(guess_w, -0.5, 1.0, 120)
errors = [average_error(p, -0.5) for p in run_path]
right = [days_right(p, -0.5) for p in run_path]
for k in (0, 1, 2, 5, 10, 20, 40, 120):
    print(f"step {k:3d}: weights ({run_path[k][0]:+.3f}, {run_path[k][1]:+.3f})  error {errors[k]:.4f}  right {right[k]}")
check("after 120 steps: error", errors[-1], 0.04142, tol=5e-6)
check("after 120 steps: days right", right[-1], 41)
check("step 20: days right", right[20], 38)

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
valley(axes[0], run_path); axes[0].set_title("the path, step size 1", fontsize=10)
axes[1].plot(errors, color=INK, lw=2); axes[1].set_xlabel("step"); axes[1].set_ylabel("average error")
diary_plot(axes[2], run_w, -0.5, title="after 120 steps: 41 of 42 right")
plt.tight_layout(); plt.show()

## Plate XV · Step size

How big a step? Too small crawls; too big zig-zags from wall to wall; bigger still throws the ball off the edge, onto the
flat plain. We count steps until the error is within 1% of the floor.

In [ ]:
def steps_to_floor(lr, steps=5000):
    w, b, path = descend(guess_w, -0.5, lr, steps)
    k = next((i for i, p in enumerate(path) if average_error(p, -0.5) < floor_err * 1.01), None)
    silent = bool(np.all(X @ w - 0.5 <= 0))
    return k, silent, path

results = {lr: steps_to_floor(lr) for lr in (0.1, 1, 3, 3.5)}
for lr, (k, silent, _) in results.items():
    print(f"step size {lr:>4}: {'reaches the floor in ' + str(k) + ' steps' if k is not None else 'never reaches the floor'}"
          f"{'; ends silent on every day' if silent else ''}")
check("step size 0.1: steps to the floor", results[0.1][0], 641)
check("step size 1: steps to the floor", results[1][0], 63)
check("step size 3: survives", results[3][1], False)
check("step size 3.5: dies (silent on every day)", results[3.5][1], True)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (lr, (k, silent, path)) in zip(axes, results.items()):
    valley(ax, path[:200])
    ax.set_title(f"step size {lr}" + (": dies" if silent else ""), fontsize=10)
plt.tight_layout(); plt.show()

Exactly where does it die? Halve the gap between a step size that survives and one that dies, forty times (*bisection*).
The textbook rule says steps are safe while smaller than 2 ÷ the valley's steepest **curvature** at the floor (how fast
the slope itself changes). The lecture's margin note compares the two.

In [ ]:
lo, hi = 3.0, 4.0
for _ in range(40):
    mid = (lo + hi) / 2
    w, _, _ = descend(guess_w, -0.5, mid, 3000)
    if np.all(X @ w - 0.5 <= 0):
        hi = mid
    else:
        lo = mid
active = (X @ floor_w - 0.5) > 0
H = 2 * (X[active].T @ X[active]) / len(X)          # the curvature at the floor
limit = 2 / np.linalg.eigvalsh(H).max()
print(f"the neuron dies for step sizes above {lo:.4f}; the textbook limit is {limit:.3f}")
check("death threshold", lo, 3.0348, tol=5e-5)
check("textbook limit, 2 ÷ largest curvature", limit, 3.065, tol=5e-4)

## Plate XVI · The flat plain: a dead neuron

At step size 3.5, three steps land the ball on the flat plain. There the neuron is silent on every day, every slope is 0,
and it can never learn again: a **dead neuron**.

How often does that happen from a random start? The lecture's margin note: in 63% of 2,000 random starts (all three
numbers drawn at random, step size 1, 5,000 steps). We run the same 2,000 starts with the same random numbers (NumPy's
generator, seed 0, used in the same order), all at once.

In [ ]:
k, silent, path = results[3.5]
w_dead = path[-1]
print(f"after step size 3.5: weights {w_dead.round(3)}; silent on every day: {silent}; slopes {average_slopes(w_dead, -0.5)[0]}")
check("the dead neuron's error", average_error(w_dead, -0.5), 18 / 42)

rng = np.random.default_rng(0)
rng.uniform(-2, 2, (2000, 2)); rng.normal(0, 1, (2000, 2))     # the draws the lecture used first, for other runs
S = rng.normal(0, 1, (2000, 3))                                 # 2,000 random starts: warmth weight, rain weight, bias
Wb, Bb = S[:, :2].copy(), S[:, 2].copy()
alive = np.ones(2000, bool)                                     # stop a start whose numbers blow up, as the lecture did
with np.errstate(over="ignore", invalid="ignore"):
  for _ in range(5000):
    T = X @ Wb.T + Bb                                           # 42 × 2000: every day, every start
    G = 2 * (np.maximum(0, T) - Y[:, None]) * (T > 0)
    Wb[alive] -= ((G.T @ X) / len(X))[alive]
    Bb[alive] -= G.mean(axis=0)[alive]
    alive &= np.isfinite(Wb).all(1) & (np.abs(Wb).max(1) <= 1e6) & (np.abs(Bb) <= 1e6)
  dead = np.all(X @ Wb.T + Bb <= 0, axis=0)
print(f"dead at the end: {dead.sum()} of 2000 ({dead.mean():.0%})")
check("random starts that die", dead.mean(), 0.63, tol=0.01)

## Plate XVII · Free the bias

The bias has a slope like any weight; its input is always 1. Two comparisons:

1. Lecture 1's hand-set neuron, graded by our answer key (output above ½): only 33 of 42.
2. Let all three numbers learn from the guess (step size 1): every day is right from step 25.

Learning finds *a* line that works, not *the* line: its numbers differ from ours.

In [ ]:
check("Lecture 1's neuron, by the answer key", days_right(np.array([2.0, -3.0]), -0.5), 33)
check("Lecture 1's neuron, by Lecture 1's reading (fires = go)", int(np.sum((X @ [2.0, -3.0] - 0.5 > 0) == (Y > 0))), 42)

w3, b3 = guess_w.copy(), -0.5
first_all_right = None
for step in range(20001):
    if first_all_right is None and days_right(w3, b3) == 42:
        first_all_right = step
    if step == 20000:
        break
    gw, gb = average_slopes(w3, b3)
    w3, b3 = w3 - gw, b3 - gb
print(f"every day right from step {first_all_right}; after 20,000 steps: weights {w3.round(4)}, bias {b3:.4f}")
check("first step with every day right", first_all_right, 25)
check("learned warmth weight", w3[0], 1.6035, tol=5e-5)
check("learned rain weight", w3[1], -2.0983, tol=5e-5)
check("learned bias", b3, 0.0490, tol=5e-5)

fig, axes = plt.subplots(1, 2, figsize=(8.6, 4.2))
diary_plot(axes[0], np.array([2.0, -3.0]), -0.5, title="Lecture 1's neuron, by the answer key: 33")
diary_plot(axes[1], w3, b3, title="all three learned: 42")
plt.tight_layout(); plt.show()

## Plates XVIII–XIX · The old rule and the new

Gradient descent (step every number against its slope) is Cauchy's idea, from 1847. Rosenblatt's perceptron (1958) learned
by a different rule (written down 1960–62): go through the days; when the neuron is wrong, **add** the day to the weights if
it should have fired, **subtract** it if it should not (and the same for the bias, with input 1).

For one neuron the old rule is safer. But it needs each neuron's right answer, and a hidden neuron has none (Lecture 3).

In [ ]:
def rosenblatt(w, b, max_passes=100):
    """Rosenblatt's error-correction rule: fix the weights on every day the neuron gets wrong (fires = go)."""
    w, b = np.array(w, float), float(b)
    passes, corrections, per_pass = 0, 0, []
    for _ in range(max_passes):
        mistakes = 0
        for x, y in zip(X, Y):
            if (x @ w + b > 0) != (y > 0):
                s = 1.0 if y > 0 else -1.0
                w, b = w + s * x, b + s
                mistakes += 1
        passes, corrections = passes + 1, corrections + mistakes
        per_pass.append(mistakes)
        if mistakes == 0:
            break
    return w, b, passes, corrections, per_pass

wp, bp, passes, corrections, per_pass = rosenblatt(guess_w, -0.5)
print(f"Rosenblatt's rule: {corrections} corrections in {passes} passes ({per_pass}); ends at {wp.round(3)}, bias {bp}")
check("corrections", corrections, 41)
check("passes", passes, 14)
check("final weights", tuple(float(v) for v in np.round(wp, 3)), (3.979, -3.926))
check("final bias", bp, -1.5)

### Your turn 3 · A nudge on the bias

Do the nudge test on the bias for the rainy day: nudge it from −0.5 to −0.499 and compute rise ÷ nudge. Does it agree with
the three-factor slope for the bias, 1.8?

*Hint:* `rainy_error` takes the bias as `b=`.

In [ ]:
bias_slope_by_nudge = None   # TODO

### Your turn 4 · Step size 5

The lecture asks: you set the step size to 5. What happens? Run it and look at the final weights and the error.

*Hint:* `descend(guess_w, -0.5, 5.0, 3000)`, then test whether the neuron is silent on every day.

In [ ]:
# TODO

### Your turn 5 · Start from nothing

Start from weights (0, 0), bias −0.5. What does gradient descent do? And Rosenblatt's rule?

*Hint:* at (0, 0) every total is −0.5. What does the gate do to the slopes?

In [ ]:
# TODO

## What we built

- A neuron learns by guessing, measuring its error, and stepping each weight against its slope.
- The slope is a chain: twice the miss, times the gate, times the input. A shut gate passes nothing.
- Over all the weights the error is a valley; learning rolls down it. The step size matters.
- A neuron silent on every day can never learn again.

**Next · Lecture 3, Many Neurons:** one line can't say "not too hot, not too cold". Two neurons can, and blame can flow
back through both.

---

## Solutions

**Your turn 1.** 0.8 + 0.1 − 0.5 = 0.4. We wanted 1: error (0.4 − 1)² = 0.36.

In [ ]:
sunny = np.array([0.8, 0.1])
sunny_output = max(0.0, sunny @ guess_w + guess_b)
sunny_error = (sunny_output - 1.0) ** 2
check("sunny day: output", sunny_output, 0.4)
check("sunny day: error", sunny_error, 0.36)

**Your turn 2.** 2 × (−0.6) × 1 × 0.8 = −0.96. Negative, so the step turns warmth up.

In [ ]:
sunny_slope_warmth = slopes_one_day(sunny, 1.0, guess_w, guess_b)[0]
check("sunny day: slope for warmth", sunny_slope_warmth, -0.96)
check("one step moves the warmth weight up", bool(guess_w[0] - 0.1 * sunny_slope_warmth > guess_w[0]), True)

**Your turn 3.** The nudge agrees with the chain, to three decimals.

In [ ]:
bias_slope_by_nudge = (rainy_error(1.0, b=-0.499) - rainy_error(1.0, b=-0.5)) / 0.001
check("bias slope by nudge", bias_slope_by_nudge, 1.8, tol=2e-3)

**Your turn 4.** The ball overshoots, lands on the flat plain, and the neuron stops learning for good.

In [ ]:
w5, _, _ = descend(guess_w, -0.5, 5.0, 3000)
print(f"step size 5 ends at {w5.round(3)}")
check("step size 5: silent on every day", bool(np.all(X @ w5 - 0.5 <= 0)), True)
check("step size 5: error", average_error(w5, -0.5), 18 / 42)

**Your turn 5.** Gradient descent never moves: the neuron is silent on every day, so every slope is 0. Rosenblatt's rule
learns from silent days too, and finishes in 12 corrections.

In [ ]:
w0, _, _ = descend([0.0, 0.0], -0.5, 1.0, 1000)
check("gradient descent from (0, 0) stays at", tuple(float(v) for v in w0), (0.0, 0.0))
_, _, _, c0, _ = rosenblatt([0.0, 0.0], -0.5)
check("Rosenblatt's rule from (0, 0): corrections", c0, 12)